# DEEPX Dual-Camera Shared YOLOv8n Fine-Tuning

이 notebook은 `YOLOV8N.onnx`의 fused weight를 `shared_backbone + front_head + top_head` 구조로 이식한 뒤, Roboflow의 front-view/top-view person 데이터셋으로 두 head를 fine-tuning하고 DEEPX용 split ONNX를 출력합니다.

출력 파일:
- `runs/dual_yolov8_person/best.pt`
- `runs/dual_yolov8_person/deepx_split_onnx/shared_backbone.onnx`
- `runs/dual_yolov8_person/deepx_split_onnx/front_head.onnx`
- `runs/dual_yolov8_person/deepx_split_onnx/top_head.onnx`

In [ ]:
# Colab dependency setup
!python -m pip install -q torch torchvision onnx onnxscript onnxruntime numpy pillow tqdm pycocotools gdown roboflow

## 1. Google Drive mount

프로젝트 폴더를 Google Drive에 올려둔 경우 먼저 Drive를 연결합니다. Colab 로컬 `/content/YOLO_optim`에 업로드해서 사용할 때도 이 셀은 실행해도 괜찮습니다.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Colab; skipping Google Drive mount.')

## 2. Project setup

`YOLO_optim` 폴더 전체를 Colab에 업로드하거나 Google Drive에 올린 뒤 실행합니다. 폴더 위치가 다르면 `YOLO_OPTIM_ROOT` 환경변수로 지정하세요.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

candidate_roots = []
if os.environ.get('YOLO_OPTIM_ROOT'):
    candidate_roots.append(Path(os.environ['YOLO_OPTIM_ROOT']))
candidate_roots.extend([
    Path.cwd(),
    Path('/content/drive/MyDrive/YOLO_optim'),
    Path('/content/drive/MyDrive/YOLO_OPTIM'),
    Path('/content/YOLO_optim'),
    Path('/content/YOLO_OPTIM'),
])

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'dual_yolov8').exists() and (root / 'YOLOV8N.onnx').exists():
        PROJECT_ROOT = root
        break
assert PROJECT_ROOT is not None, 'YOLO_optim root not found. Set YOLO_OPTIM_ROOT to the folder containing YOLOV8N.onnx and dual_yolov8/.'

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)
print('CUDA available =', __import__('torch').cuda.is_available())

## 3. Original ONNX check

`YOLOV8N.onnx`가 YOLOv8n fused export인지 확인하고, PyTorch 재구현과 원본 ONNX 출력이 맞는지 비교합니다.

In [ ]:
ONNX_PATH = PROJECT_ROOT / 'YOLOV8N.onnx'
assert ONNX_PATH.exists(), f'Missing {ONNX_PATH}'
!python -m dual_yolov8.inspect_onnx YOLOV8N.onnx
!python -m dual_yolov8.parity_check --onnx YOLOV8N.onnx

## 4. Roboflow datasets

Front-view는 요청한 `object-detection-i9fzg/person-detection-tfsgp/3` 데이터셋을 사용합니다.

Top-view는 `yolo-k9iku/cctv-indoor-person-detection-bi2vp/1` 데이터셋을 사용합니다. API key는 notebook에 직접 저장하지 않도록 Colab Secrets 또는 환경변수 `ROBOFLOW_TOP_API_KEY`로 넣는 방식을 기본으로 했습니다.

In [ ]:
from roboflow import Roboflow

DATA_ROOT = PROJECT_ROOT / 'datasets'
ROBOFLOW_ROOT = DATA_ROOT / 'roboflow_raw'
ROBOFLOW_ROOT.mkdir(parents=True, exist_ok=True)

FRONT_ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_FRONT_API_KEY', 'unauthorized')
TOP_ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_TOP_API_KEY')
if not TOP_ROBOFLOW_API_KEY:
    try:
        from google.colab import userdata
        TOP_ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_TOP_API_KEY')
    except Exception:
        TOP_ROBOFLOW_API_KEY = None
assert TOP_ROBOFLOW_API_KEY, 'Set ROBOFLOW_TOP_API_KEY in Colab Secrets or environment variables before running this cell.'

def download_roboflow_dataset(api_key, workspace, project_name, version):
    previous = Path.cwd()
    os.chdir(ROBOFLOW_ROOT)
    try:
        rf = Roboflow(api_key=api_key)
        project = rf.workspace(workspace).project(project_name)
        dataset = project.version(version).download('yolov8')
        return Path(dataset.location)
    finally:
        os.chdir(previous)

FRONT_DATASET_ROOT = download_roboflow_dataset(
    FRONT_ROBOFLOW_API_KEY,
    workspace='object-detection-i9fzg',
    project_name='person-detection-tfsgp',
    version=3,
)
TOP_DATASET_ROOT = download_roboflow_dataset(
    TOP_ROBOFLOW_API_KEY,
    workspace='yolo-k9iku',
    project_name='cctv-indoor-person-detection-bi2vp',
    version=1,
)

print('FRONT_DATASET_ROOT =', FRONT_DATASET_ROOT)
print('TOP_DATASET_ROOT =', TOP_DATASET_ROOT)

## 5. Dataset path check

In [ ]:
def yolo_split_paths(dataset_root, split='train'):
    root = Path(dataset_root)
    images = root / split / 'images'
    labels = root / split / 'labels'
    assert images.exists(), f'Missing images path: {images}'
    assert labels.exists(), f'Missing labels path: {labels}'
    return images, labels


def optional_yolo_split_paths(dataset_root, preferred=('valid', 'test')):
    root = Path(dataset_root)
    for split in preferred:
        images = root / split / 'images'
        labels = root / split / 'labels'
        if images.exists() and labels.exists() and any(images.glob('*')):
            return split, images, labels
    return None, None, None


FRONT_IMAGES, FRONT_LABELS = yolo_split_paths(FRONT_DATASET_ROOT, 'train')
TOP_IMAGES, TOP_LABELS = yolo_split_paths(TOP_DATASET_ROOT, 'train')
FRONT_VAL_SPLIT, FRONT_VAL_IMAGES, FRONT_VAL_LABELS = optional_yolo_split_paths(FRONT_DATASET_ROOT)
TOP_VAL_SPLIT, TOP_VAL_IMAGES, TOP_VAL_LABELS = optional_yolo_split_paths(TOP_DATASET_ROOT)

FRONT_TOTAL_IMAGES = len(list(FRONT_IMAGES.glob('*')))
FRONT_TOTAL_LABELS = len(list(FRONT_LABELS.glob('*.txt')))
TOP_TOTAL_IMAGES = len(list(TOP_IMAGES.glob('*')))
TOP_TOTAL_LABELS = len(list(TOP_LABELS.glob('*.txt')))

print('front train images:', FRONT_TOTAL_IMAGES)
print('front train labels:', FRONT_TOTAL_LABELS)
print('top train images:', TOP_TOTAL_IMAGES)
print('top train labels:', TOP_TOTAL_LABELS)
print('front val split:', FRONT_VAL_SPLIT, FRONT_VAL_IMAGES)
print('top val split:', TOP_VAL_SPLIT, TOP_VAL_IMAGES)


## 6. Fine-tuning sample count

`None`이면 전체 train 이미지를 사용합니다. 빠른 테스트를 하고 싶으면 정수로 바꾸세요.

In [ ]:
# 전체 학습: None
# 예시: front 500장, top 500장만 학습하려면 각각 500으로 설정
MAX_FRONT_SAMPLES = None
MAX_TOP_SAMPLES = None

# MAX_FRONT_SAMPLES = 500
# MAX_TOP_SAMPLES = 500

front_used = FRONT_TOTAL_IMAGES if MAX_FRONT_SAMPLES is None else min(MAX_FRONT_SAMPLES, FRONT_TOTAL_IMAGES)
top_used = TOP_TOTAL_IMAGES if MAX_TOP_SAMPLES is None else min(MAX_TOP_SAMPLES, TOP_TOTAL_IMAGES)
print(f'front fine-tuning images: {front_used} / {FRONT_TOTAL_IMAGES}')
print(f'top fine-tuning images: {top_used} / {TOP_TOTAL_IMAGES}')

## 7. Model smoke test

이 단계에서 모델 구조는 이미 `shared_backbone + front_head + top_head`이고, forward는 두 출력을 반환합니다.

In [ ]:
import torch
from dual_yolov8.model import DualYolov8n, load_fused_yolov8n_from_onnx

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DualYolov8n(nc=1).to(device)
report = load_fused_yolov8n_from_onnx(model, ONNX_PATH, strict=True)
print('copied tensors:', len(report.copied))
print('person cls sliced tensors:', len(report.reshaped))
model.eval()
with torch.no_grad():
    front_output, top_output = model(
        torch.zeros(1, 3, 640, 640, device=device),
        torch.zeros(1, 3, 640, 640, device=device),
    )
print('front_output:', tuple(front_output.shape))
print('top_output:', tuple(top_output.shape))

## 8. Site fine-tuning

Roboflow view 데이터와 실제 설치 장소 calibration 데이터를 섞어 학습하고, 매 epoch마다 calibration valid/test 성능을 저장합니다.


In [ ]:
IMG_SIZE = 640
BATCH = 4
WORKERS = 2
SITE_OUT = PROJECT_ROOT / 'runs' / 'dual_yolov8_site_mix'
CALIB_FRONT_ROOT = PROJECT_ROOT / 'calibration' / 'labeled' / 'front'
CALIB_TOP_ROOT = PROJECT_ROOT / 'calibration' / 'labeled' / 'top'

assert CALIB_FRONT_ROOT.exists(), f'Missing calibration front dataset: {CALIB_FRONT_ROOT}'
assert CALIB_TOP_ROOT.exists(), f'Missing calibration top dataset: {CALIB_TOP_ROOT}'

# 빠른 smoke test가 필요하면 아래 값을 작게 설정하세요. 전체 학습은 None을 유지합니다.
SITE_MAX_FRONT_ROBOFLOW = MAX_FRONT_SAMPLES
SITE_MAX_TOP_ROBOFLOW = MAX_TOP_SAMPLES
SITE_MAX_FRONT_CALIB = None
SITE_MAX_TOP_CALIB = None
SITE_MAX_FRONT_EVAL = None
SITE_MAX_TOP_EVAL = None

cmd = [
    sys.executable, '-m', 'dual_yolov8.site_finetune',
    '--onnx', str(ONNX_PATH),
    '--front-roboflow-root', str(FRONT_DATASET_ROOT),
    '--top-roboflow-root', str(TOP_DATASET_ROOT),
    '--front-calib-root', str(CALIB_FRONT_ROOT),
    '--top-calib-root', str(CALIB_TOP_ROOT),
    '--out', str(SITE_OUT),
    '--imgsz', str(IMG_SIZE),
    '--batch', str(BATCH),
    '--workers', str(WORKERS),
    '--warmup-epochs', '1',
    '--mixed-finetune-epochs', '6',
    '--site-finetune-epochs', '4',
    '--lr', '1e-4',
    '--site-lr', '2.5e-5',
    '--val-conf', '0.001',
    '--nms-iou', '0.7',
    '--opset', '17',
]
for flag, value in [
    ('--max-front-roboflow-samples', SITE_MAX_FRONT_ROBOFLOW),
    ('--max-top-roboflow-samples', SITE_MAX_TOP_ROBOFLOW),
    ('--max-front-calib-samples', SITE_MAX_FRONT_CALIB),
    ('--max-top-calib-samples', SITE_MAX_TOP_CALIB),
    ('--max-front-eval-samples', SITE_MAX_FRONT_EVAL),
    ('--max-top-eval-samples', SITE_MAX_TOP_EVAL),
]:
    if value is not None:
        cmd += [flag, str(value)]

print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 9. Site metrics graph

Calibration test 기준 AP50/mAP50-95 변화 그래프와 epoch별 CSV를 확인합니다.


In [ ]:
from IPython.display import HTML, Image as DisplayImage, display
import csv
import html

metrics_csv = SITE_OUT / 'metrics_history.csv'
plot_png = SITE_OUT / 'calibration_test_metrics.png'
assert metrics_csv.exists(), metrics_csv
assert plot_png.exists(), plot_png

display(DisplayImage(filename=str(plot_png)))
rows = list(csv.DictReader(metrics_csv.open()))
shown = rows[-12:]
headers = list(shown[0].keys()) if shown else []
html_rows = ['<table><thead><tr>' + ''.join(f'<th>{html.escape(h)}</th>' for h in headers) + '</tr></thead><tbody>']
for row in shown:
    html_rows.append('<tr>' + ''.join(f'<td>{html.escape(row[h])}</td>' for h in headers) + '</tr>')
html_rows.append('</tbody></table>')
display(HTML(''.join(html_rows)))


## 10. ONNXRuntime verification

`best_val.pt`에서 export된 split ONNX 3개가 PyTorch checkpoint와 같은 출력을 내는지 확인합니다.


In [ ]:
!python -m dual_yolov8.verify_split_onnx   --checkpoint {SITE_OUT}/best_val.pt   --onnx-dir {SITE_OUT}/deepx_split_onnx   --imgsz 640


## 11. Calibration test comparison

원본 YOLOv8n ONNX와 site fine-tuned split ONNX를 실제 설치 장소 calibration test split에서 비교합니다.


In [ ]:
FRONT_EVAL_IMAGES = CALIB_FRONT_ROOT / 'images' / 'test'
FRONT_EVAL_LABELS = CALIB_FRONT_ROOT / 'labels' / 'test'
TOP_EVAL_IMAGES = CALIB_TOP_ROOT / 'images' / 'test'
TOP_EVAL_LABELS = CALIB_TOP_ROOT / 'labels' / 'test'

for path in [FRONT_EVAL_IMAGES, FRONT_EVAL_LABELS, TOP_EVAL_IMAGES, TOP_EVAL_LABELS]:
    assert path.exists(), path

print('front eval:', FRONT_EVAL_IMAGES)
print('top eval:', TOP_EVAL_IMAGES)


In [ ]:
EVAL_MAX_IMAGES = None
EVAL_CONF = 0.001
EVAL_NMS_IOU = 0.7

cmd = [
    sys.executable, '-m', 'dual_yolov8.evaluate_onnx',
    '--original-onnx', str(ONNX_PATH),
    '--split-onnx-dir', str(SITE_OUT / 'deepx_split_onnx'),
    '--front-images', str(FRONT_EVAL_IMAGES),
    '--front-labels', str(FRONT_EVAL_LABELS),
    '--top-images', str(TOP_EVAL_IMAGES),
    '--top-labels', str(TOP_EVAL_LABELS),
    '--imgsz', '640',
    '--conf', str(EVAL_CONF),
    '--nms-iou', str(EVAL_NMS_IOU),
]
if EVAL_MAX_IMAGES is not None:
    cmd += ['--max-images', str(EVAL_MAX_IMAGES)]

print(' '.join(cmd))
result = subprocess.run(cmd, check=True, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
